# Columnas y Expresiones

Iniciar Sesión de Spark (Spark Session)

---

In [ ]:
from pyspark.sql import SparkSession

# crear la sesión
spark = SparkSession \
        .builder \
        .appName("DataFrames Basics") \
        .master("local[*]") \
        .getOrCreate()

spark.version

In [ ]:
spark

In [ ]:
# Para optimización de conversión a Pandas
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [ ]:
# Importar funciones sql
from pyspark.sql.functions import *

Leer archivo JSON

In [ ]:
cochesDF = spark.read \
    .option("inferSchema", True) \
    .json("cars_dates.json")

## Ejemplos

Seleccionar una columna

In [ ]:
cochesDF.select(col("Cylinders")).show(3, False)

Podemos utilizar varios métodos para referirnos a una columna

In [ ]:
# Varios metodos de select
cochesDF.select(
    cochesDF.Name,
    col("Miles_per_Gallon"),
    "Displacement"
).show(3)

Expresiones. Podemos utilizzar el lenguaje SQL dentro de un select como expresiones para trabajar y transformar columnas

In [ ]:
cochesenKgDF = cochesDF.select(
    col("Name"),
    col("Horsepower"),
    (col("Weight_in_lbs")/2.2).cast("int").alias("Weight_in_kg_2"), #casteo el resultado a un int
    expr("Weight_in_lbs / 1000").cast("string").alias("Weight_in_T") #casteo el resultado a un str
)
cochesenKgDF.printSchema()
cochesenKgDF.show(3)

In [ ]:
# trabajando con expressions
cochesConSelectExpresionDF = cochesDF.selectExpr(
    "Name",
    "Weight_in_lbs",
    "Weight_in_lbs / 2.2"
  )
cochesConSelectExpresionDF.show(3)

### Procesamiento de DFs

Añadir una columna

In [ ]:
cochesNuevaColumnaDF = cochesDF.withColumn("Weight_in_kg_3", col("Weight_in_lbs") / 2.2)
cochesNuevaColumnaDF.show(3)

Renombrar una columna

In [ ]:
cochesColumnaRenombradaDF = cochesDF.withColumnRenamed("Weight_in_lbs", "Weight in pounds")
cochesColumnaRenombradaDF.show(3)

In [ ]:
# as we hace special characters (spaces) we have to use the ``
cochesColumnaRenombradaDF.selectExpr("`Weight in pounds`").show(3)

Eliminar una columna

In [ ]:
cochesColumnaRenombradaDF.printSchema()

In [ ]:
eliminarColsDF = cochesColumnaRenombradaDF.drop("Horsepower", "Displacement")
eliminarColsDF.printSchema()


Filtrar DF

In [ ]:
filtroCochesDF = cochesDF.filter(col("Origin") != "USA")
filtroCochesDF2 = cochesDF.where(col("Origin") != "USA")
filtroCochesDF.show(3)
print(f"{filtroCochesDF.count()} == {filtroCochesDF2.count()}")

In [ ]:
# Filtrar con expressions strings
cochesUSADF = cochesDF.filter("Year='01-01-1970'")
cochesUSADF.show(3)

Filtros de cadena

In [ ]:
cochesPotenciaDF = cochesDF.filter(col("Origin") == "USA").filter(col("Horsepower") > 150)
cochesPotenciaDF2 = cochesDF.filter((col("Origin") == "USA") & (col("Horsepower") > 150))
cochesPotenciaDF3 = cochesDF.filter("Origin = 'USA' and Horsepower > 150")
cochesPotenciaDF.show(3)
cochesPotenciaDF2.show(3)
cochesPotenciaDF3.show(3)

## Ejercicios

1. Lee el archivo de deptmanagers.csv y haz un select de las 2 columnas que quieras
   
2. Añade una columna nueva al dataset de movies.json que sea el total generado por cada pelicula = US_Gross + Worldwide_Gross + DVD sales. Estas obteniendo algun nulo? Como lo puedes solucionar?
   
3. Selecciona todas las peliculas que tengan una nota mayor que 7 o igual en IMDB (IMDB_Rating), intenta hacerlo de todas las maneras que sepas.

Ejercicio 1

Ejercicio 2

Ejercicio 3